# Instrument the seed-42 Transformer repeat

Question: which participants, classes, and probability changes contribute to the validation-loss rebounds?
Repeat folds 01 and 04 from the original five-fold plan, with the same seed-42 dropout-0.2 recipe.
Keep context thirty, width sixteen, Adam 0.001, forty epochs, patience twelve, and macro-F1 checkpoint selection fixed.
Only evidence collection changes. The maximum budget is two fits and eighty epochs.

Save validation probabilities and participant/class summaries each epoch. Retain intermediate checkpoints for epochs thirteen through seventeen when reached.
The three final test participants remain reserved. These two selected folds do not estimate performance across all 26 development participants.
Read [the diagnostic protocol](../docs/transformer-diagnostics.md) before training.

Completed results: docs/experiments/transformer-diagnostics-2026-11-12/README.md


## Locate the repository and set execution switches

**What:** Find the local checkout, configure logging, and declare setup, preparation, preflight, and training switches.
**Why:** Keep paths portable and make optimization an explicit choice. Only the no-gradient preflight defaults to enabled.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("transformer_diagnostics_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-diagnostics-grouped.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/transformer-diagnostics"  # Select the PR branch before merge
RUN_SETUP = False
RUN_PREPARATION = False
RUN_CHECK = True
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)
logger.info("Setup=%s preparation=%s preflight=%s training=%s",
            RUN_SETUP, RUN_PREPARATION, RUN_CHECK, RUN_TRAINING)


## Optional environment setup

**What:** Optionally clone this branch and install locked dependencies with uv.
**Why:** Support a fresh machine without environment-specific branches. Existing local checkouts can leave RUN_SETUP disabled.
Prepared feature data must also be available; cloning does not download or copy the dataset.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Verify the unchanged recipe and selected folds

**What:** Compare the diagnostic recipe and participant assignments with the original seed-42 dropout-0.2 experiment.
**Why:** Instrumentation must not silently change the optimization settings or create a new two-fold split.
The five-fold split is computed first. The run selects its original folds 01 and 04.


In [ ]:
import pandas as pd

from sleep_classifier.grouped_validation import GroupedValidationExperiment

experiment = GroupedValidationExperiment(ROOT / "configs/transformer-diagnostics-grouped.yaml")
reference = GroupedValidationExperiment(
    ROOT / "configs/transformer-dropout-grouped-validation.yaml"
)
plan = experiment.plan()
reference_plan = reference.plan()
expected = [fold for fold in reference_plan["folds"] if fold["name"] in {"fold-01", "fold-04"}]
if plan["folds"] != expected:
    raise ValueError("Diagnostic folds differ from the original five-fold assignments")
recipe = {key: value for key, value in experiment.model_settings.items()
          if key not in {"diagnostics", "output_dir"}}
reference_recipe = {key: value for key, value in reference.model_settings.items()
                    if key != "output_dir"}
if recipe != reference_recipe:
    raise ValueError("Diagnostic recipe differs from the measured seed-42 candidate")
if len(plan["folds"]) != 2 or plan["maximum_epochs"] != 80:
    raise ValueError("Diagnostic work exceeds the agreed two-fit budget")
display(pd.DataFrame(plan["folds"]).set_index("name"))
display(pd.DataFrame([experiment.model_settings["diagnostics"]]))
logger.info("Plan verified: two original folds; maximum %s epochs", plan["maximum_epochs"])
logger.info("Subset validation only; the final test participants remain reserved")


## Prepare the selected folds once

**What:** Optionally copy validated development features and freeze the two fold configurations.
**Why:** Keep the original split, checksums, and training-only scaling contract without modifying prior experiment outputs.
Leave RUN_PREPARATION disabled when this experiment already has a manifest.
Preparation refuses an existing output directory and does not train a model.


In [ ]:
if RUN_PREPARATION:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
        --config configs/transformer-diagnostics-grouped.yaml --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Preparation failed; inspect the output directory before retrying")
else:
    logger.info("Preparation skipped")


## Check both prepared models without optimization

**What:** Run the no-gradient preflight against each actual prepared fold configuration.
**Why:** Verify inputs, scaling, tensor shapes, parameter count, and device before starting the repeat.
The check fits preprocessing on training participants only. It writes no model or training evidence.


In [ ]:
if RUN_CHECK and (experiment.output_dir / "manifest.json").exists():
    %cd "{ROOT}"
    for fold in plan["folds"]:
        fold_config = experiment.output_dir / fold["name"] / "gru.yaml"
        logger.info("Checking %s without optimization", fold["name"])
        !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
            --config "{fold_config}" --check
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Diagnostic preflight failed")
elif RUN_CHECK:
    logger.info("Preflight pending: prepare the two folds first")
else:
    logger.info("Preflight skipped")


## Train the instrumented repeat when ready

**What:** Optionally train fresh seed-42 models for the two prepared folds.
**Why:** Collect the missing per-epoch evidence while keeping the existing model, loss, and checkpoint-selection rule fixed.
Enable RUN_TRAINING only when ready. The first failure stops execution and retains completed evidence.
Existing results are refused; automatic resume is unsupported. Inspect a partial run before choosing a recovery action.


In [ ]:
if RUN_TRAINING:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.grouped_validation \
        --config configs/transformer-diagnostics-grouped.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Training failed; inspect retained evidence before retrying")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")


## Inspect available evidence and run status

**What:** Read the local experiment evidence without fitting a model or changing saved training artifacts.
**Why:** Distinguish prepared, running, and completed folds. Partial evidence can still explain a failure or interrupted run.
The evidence reader verifies artifact hashes and reports missing data rather than inventing scores.


In [ ]:
from sleep_classifier.experiment_evidence import EpochEvidenceReport

reporter = EpochEvidenceReport(experiment.output_dir)
evidence = reporter.build()
status_rows = [{"fold": fold["fold"], "status": fold["status"],
                "recorded_epochs": len(fold["epochs"])} for fold in evidence["folds"]]
if status_rows:
    display(pd.DataFrame(status_rows).set_index("fold"))
else:
    logger.info("No prepared diagnostic folds are available yet")


## Compare matched loss curves and optimizer records

**What:** Display per-epoch losses, macro-F1, gradient norms, and actual learning rates for each available fold.
**Why:** Separate a confidence-loss increase from a hard-label change or an optimizer event.
NLL is the mean negative logarithm of the correct-class probability. Lower is better; confident errors contribute larger losses.
Training-evaluation NLL and validation NLL both use evaluation mode at fixed end-of-epoch weights.
Ordinary training loss includes dropout and changing weights. Its gap with validation loss needs separate interpretation.
The gradient norm is measured before clipping; the optimizer still applies the unchanged clipping threshold.


In [ ]:
import matplotlib.pyplot as plt

epoch_rows = [{"fold": fold["fold"], **epoch}
              for fold in evidence["folds"] for epoch in fold["epochs"]]
if epoch_rows:
    columns = ["fold", "epoch", "validation_nll", "validation_criterion_nll", "validation_macro_f1",
               "training_evaluation_nll", "training_stochastic_nll", "learning_rates",
               "gradient_norm_pre_clip_mean", "gradient_norm_pre_clip_max"]
    display(pd.DataFrame(epoch_rows).reindex(columns=columns).set_index(["fold", "epoch"]))
    figure = reporter.plot_curves()
    display(figure)
    plt.close(figure)
else:
    logger.info("Per-epoch curves are pending the local training run")


## Inspect stage scores and confident errors

**What:** Display per-class NLL, precision, recall, and F1 near the inspected epochs.
Also show confidence on errors and the fraction of all targets receiving high-confidence wrong labels.

**Why:** A probability-loss rise can accompany improving hard labels.
These views show stage trade-offs and confidence changes before we attribute a rebound to a particular participant or class.
The complete epoch evidence remains available outside this display window.

Stage indices are 0 = Wake, 1 = Light, 2 = Deep, and 3 = REM.


In [ ]:
class_rows = []
confidence_rows = []
for fold in evidence["folds"]:
    for epoch in fold["epochs"]:
        if not 10 <= epoch["epoch"] <= 18:
            continue
        identity = {"fold": fold["fold"], "epoch": epoch["epoch"]}
        confidence_rows.append({
            **identity, "mean_confidence_wrong": epoch["mean_confidence_wrong"],
            "high_confidence_error_fraction": epoch["high_confidence_error_fraction"],
            "clipped_step_fraction": epoch["clipped_step_fraction"],
        })
        class_rows.extend({**identity, "stage": stage, **scores}
                          for stage, scores in epoch["validation_classes"].items())
if class_rows:
    display(pd.DataFrame(class_rows).set_index(["fold", "epoch", "stage"]))
    display(pd.DataFrame(confidence_rows).set_index(["fold", "epoch"]))
else:
    logger.info("Class and confidence evidence for epochs 10 through 18 is not available yet")


## Attribute loss changes near epoch fourteen

**What:** Compare saved predictions for consecutive epochs thirteen through seventeen when both epochs are available.
**Why:** Identify the participants, true classes, and individual targets that contribute to each NLL change.
A target can keep the same predicted label while its correct-class probability shrinks and its NLL increases.
Participant and class contributions decompose the same total change; they are separate views and must not be added together.
These diagnostics explain observed errors. They do not change checkpoint selection or prove a causal mechanism.
An early stop can leave some comparisons unavailable; do not extend the budget just to obtain them.


In [ ]:
for fold in evidence["folds"]:
    available_epochs = {row["epoch"] for row in fold["epochs"]}
    for before, after in [(13, 14), (14, 15), (15, 16), (16, 17)]:
        if not {before, after}.issubset(available_epochs):
            logger.info("%s comparison %s -> %s is unavailable", fold["fold"], before, after)
            continue
        transition = reporter.compare_epochs(fold["fold"], before, after)
        logger.info("%s epoch %s -> %s: NLL difference %.6f",
                    fold["fold"], before, after, transition["nll_difference"])
        display(pd.DataFrame(transition["participants"]))
        display(pd.DataFrame(transition["classes"]))
        display(pd.DataFrame(transition["largest_targets"]))


## Save the local evidence report

**What:** Write the evidence summary and available diagnostic figures through the reproducible reporting CLI.
**Why:** Keep inspection available outside this notebook and retain a local record of partial or completed results.
This operation reads evidence only. It does not train, select another checkpoint, or overwrite per-epoch training artifacts.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python -m sleep_classifier.experiment_evidence \
    --output-dir artifacts/transformer-diagnostics --report
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Evidence report failed")


## Export the executed HTML report

**What:** Execute report cells in a fresh kernel and save one standalone HTML report.
**Why:** Share the current evidence without committing large notebook outputs or accidentally repeating training.
The exporter skips setup, preparation, training, and this export cell regardless of their interactive switches.
The source notebook remains output-free in Git. The executed HTML stays under artifacts/notebooks.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/20_transformer_diagnostics.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Diagnostic notebook export failed")
